<a href="https://colab.research.google.com/github/mjmj002/Sparta/blob/main/Python/MCP/day1_MCP_ToolCalling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 1 — MCP 개념 + Tool Calling (노트북)

이 노트북은 **MCP를 '서버 구현'까지 깊게 가지 않고**,

 => ai 모델 + 맥락 + 프로토콜(통신방법) 결합적인게 MCP

**Tool Calling의 핵심 개념**을 *파이썬으로 체험*하는 실습입니다.

## 오늘 목표
- Tool(함수)을 **등록**하고 / 툴 -> 함수
- 사용자의 요청을 보고 **어떤 Tool을 쓸지 선택**(AI 가 선택)하고
- Tool 결과를 받아 **최종 답변을 생성**하는 흐름을 이해한다
- 예) 서치 -> 검색을 하는 툴

> 💡 실제 LLM(OpenAI 등)을 쓰면 더 현실적이지만,  
> 수업 안정성을 위해 **로컬에서 동작하는 간단 Router(Fake LLM)** 도 함께 제공합니다.



In [ ]:
# ✅ 기본 설정
# re : 정규표현식(파싱할 때 사용, 글자나 숫자나 스트링이나 문제열에서 내가 원하는 (매칭되는) 문자, 숫자를 찾아올 때 사용)
import os, re, json, math
from dataclasses import dataclass
from typing import Callable, Any, Dict, Optional, Tuple

print("Ready ✅")

Ready ✅


## 1) Tool(함수) 정의하기

Tool은 아래 3가지를 갖습니다.
- `name`: 도구 이름
- `description`: 도구 설명(LLM이 선택하도록 돕는 정보)
- `fn`: 실행할 파이썬 함수


In [ ]:
@dataclass # 안에 기능은 없지만 어떤 기능이 들어올 지 정리해주는 데이터 클래스(이 안에는 생성자, 함수도 없음. 네임과 스트링이라는 것만 지정되어 있음)
class Tool:
    name: str
    description: str
    fn: Callable[..., Any] # 부를 수 있는 것들만. 어떤 것도 들어올 수 있다.(Any) -> MLOps 들이 많이 사용함
                           # 부를 수 있는 함수가 여러개 들어간다. 데이터 타입이나 아무거나 올 수 있음
"""
TOOLS = {
  [str(이건 글자),Tool{name:str, description:str, fn[ex1, ex2, any]}]

}
"""
# 데이터 클래스를 받았기 때문에 이 안에 이런식으로 구성이 됐다, 라는 걸 뜻함
# TOOLS 라는 데이터가 이런식으로 구성이 된다.
TOOLS: Dict[str, Tool] = {}

# 여러개의 툴 중에서 하나를 선택하여, tool 하나만 반환한다.
def register_tool(tool: Tool): # tool 이라는 어트리뷰트를 받는데 이 툴은 Tool 데이터클래스로 지정이 됨
    TOOLS[tool.name] = tool # TOOLS 중 name 하나를 반환받음.(여러개 툴즈 반환하는데 그 중 하나를 반환함)
    return tool

# 리스트 툴즈 안에는 툴즈가 어떤 값들을 가지고 있고, 도구 상자(TOOLS)에 들어있는 모든 도구(t)를 하나씩 꺼내서 "이름: 설명" 형태의 리스트로 만듦
# TOOLS 라는 도구 모음에서 각 TOOL의 이름 설명을 반환하는게 list_tools 라는 함수이다. -> (프린트 하는 함수)
def list_tools() -> str:
    return "\n".join([f"- {t.name}: {t.description}" for t in TOOLS.values()])

print("Tool registry ready ✅")

Tool registry ready ✅


## 2) 예제 Tool 2개 등록하기

- 계산기 `calc`
- 간단한 날짜/시간 `now` (로컬 시스템 시간)


In [ ]:
from datetime import datetime, timezone, timedelta

# decorator python @ 데코레이터 : 이전에 정의된 함수나 클래스를 자동으로 넘겨서 처리.
# register_tool(Tool()

#)) -> 데코 없으면 이 형태
# register_tool 위에 툴들 중에서 이름을 툴즈에 저장하는. 양식에 맞춰서
# 메스에 맞는 기능들을 리스트에 넣을거다.

# Tool(위에 정의된 함수) 형식에 맞춰서 1번 name 이름을 정의하고
# Tool 형식에 맞춰서 2번 description 설명을 정의하고
# Tool 형식에 맞춰서 3번 fn 기능들을 정의한다.
#@register_tool(Tool(
register_tool(Tool(
    name="calc",
    description="수식을 계산합니다. 예: '2*(3+4)'",
    fn=lambda expr: eval(expr, {"__builtins__": {}}, {"math": math}) # eval함수 값들만 리스트화 해서 함수에 들어감. (문자열로 된 수식(예: "2+3")을 실제 숫자로 계산해 주는 같은 함수)
                                                                     # 라이브러리로 가져온 math의 기능들을 넣겠다.
                                                                     # __builtins__ : 파이썬의 기본 기능. 시스템상에 넣어놓겠다
))
class _Calc:  # dummy class to keep decorator usage consistent
    pass

# 시간 관련된 도구들이 나올 것
#@register_tool(Tool(
register_tool(Tool(
    name="now",
    description="현재 날짜/시간을 문자열로 반환합니다.",
    fn=lambda: datetime.now().strftime("%Y-%m-%d %H:%M:%S")
))
class _Now:
    pass

print(list_tools())

# 코랩 데코레이터 오류남 -> 빼고 실습

- calc: 수식을 계산합니다. 예: '2*(3+4)'
- now: 현재 날짜/시간을 문자열로 반환합니다.


## 3) (LLM 대신) 간단 Router 만들기

실제 LLM은 *설명*을 보고 Tool을 선택하지만,  
오늘은 수업 안정성을 위해 **규칙 기반 Router**를 씁니다.

- 수식이 보이면 `calc`
- "지금", "현재 시간"이 보이면 `now`
- 그 외는 Tool 없이 일반 답변


In [ ]:
# 라우터 : 키워드 있으면 불러오겠다.
# 노선을 정해주는 역할
def route(user_text: str) -> Tuple[Optional[str], Dict[str, Any]]:
    text = user_text.strip()

    # 매우 단순한 휴리스틱(수업용)
    if re.search(r"\d", text) and any(op in text for op in ["+", "-", "*", "/", "(", ")"]): # 계산하는 수식들 중 하나라도 있으면 함수를 불러오겠다.
        # 수식만 추출해보기(대충)
        expr = re.sub(r"[^0-9\+\-\*\/\(\)\.]", "", text)
        return "calc", {"expr": expr}

    if any(k in text for k in ["지금", "현재", "시간", "몇 시"]): # 시간 관련이 하나라도 있으면 now 반환. 아니면 도구 호출 안함
        return "now", {}

    return None, {}

# 간단한 설명
# route 에서 걸린 return 값에 해당하는 TOOL이 있으면, TOOLS 에서 이름을 불러와서 하나를 사용해라.
def tool_call(tool_name: str, args: Dict[str, Any]) -> Any:
    print(f"tool_name : {tool_name}, args : {args}")
    tool = TOOLS[tool_name]
    print(f"호출한 tool은? : {tool}")
    return tool.fn(**args) if args else tool.fn()

# 툴을 사용했으면, 툴을 사용한 결과를 반환.
# 툴을 사용하지 않았으면, 툴을 사용하지 않을 결과만 반환
def answer(user_text: str) -> str:
    tool_name, args = route(user_text)
    print(f"정답 부분의 중간 값은? : {tool_name}, {args}")
    if tool_name is None:
        return f"(Tool 없이 답변) 요청을 이해했습니다: {user_text}"

    result = tool_call(tool_name, args)
    return f"(Tool 사용: {tool_name}) 결과: {result}"

# 빠른 테스트
tests = [
    "2*(3+4) 계산해줘",
    "지금 몇 시야?",
    "파이썬이 뭐야?"
]
for t in tests:
    print("Q:", t)
    print("A:", answer(t))
    print("---")

Q: 2*(3+4) 계산해줘
정답 부분의 중간 값은? : calc, {'expr': '2*(3+4)'}
tool_name : calc, args : {'expr': '2*(3+4)'}
호출한 tool은? : Tool(name='calc', description="수식을 계산합니다. 예: '2*(3+4)'", fn=<function <lambda> at 0x7d405e7a4ea0>)
A: (Tool 사용: calc) 결과: 14
---
Q: 지금 몇 시야?
정답 부분의 중간 값은? : now, {}
tool_name : now, args : {}
호출한 tool은? : Tool(name='now', description='현재 날짜/시간을 문자열로 반환합니다.', fn=<function <lambda> at 0x7d405e7a4180>)
A: (Tool 사용: now) 결과: 2026-04-13 11:53:04
---
Q: 파이썬이 뭐야?
정답 부분의 중간 값은? : None, {}
A: (Tool 없이 답변) 요청을 이해했습니다: 파이썬이 뭐야?
---


##  실습 1 — Tool 하나 더 추가하기

아래 요구를 만족하는 Tool을 추가하세요.

### 요구사항
- Tool 이름: `text_stats`
- 설명: "문자열의 글자 수/단어 수를 계산합니다."
- 입력: `text` (문자열)
- 출력: `{"chars": ..., "words": ...}` 형태의 dict

그리고 Router에 아래 규칙을 추가하세요.
- "글자", "단어", "길이" 같은 키워드가 있으면 `text_stats`를 호출


In [ ]:

#@register_tool(Tool(
register_tool(Tool(
    name="text_stats",
    description="문자열의 글자 수/단어 수를 계산합니다.",
    fn=lambda text: {"chars": len(text), "words": len([w for w in text.split() if w])}
))
class _TextStats:
    pass

def route_v2(user_text: str) -> Tuple[Optional[str], Dict[str, Any]]:
    text = user_text.strip()

    if any(k in text for k in ["글자", "단어", "길이"]):
        # 따옴표가 있으면 그 안을 우선 사용
        m = re.search(r"['\"](.+?)['\"]", text)
        payload = m.group(1) if m else text
        return "text_stats", {"text": payload}

    if re.search(r"\d", text) and any(op in text for op in ["+", "-", "*", "/", "(", ")"]):
        expr = re.sub(r"[^0-9\+\-\*\/\(\)\.]", "", text)
        return "calc", {"expr": expr}

    if any(k in text for k in ["지금", "현재", "시간", "몇 시"]):
        return "now", {}

    return None, {}

def answer_v2(user_text: str) -> str:
    tool_name, args = route_v2(user_text)
    if tool_name is None:
        return f"(Tool 없이 답변) 요청을 이해했습니다: {user_text}"

    result = tool_call(tool_name, args)
    return f"(Tool 사용: {tool_name}) 결과: {result}"

print(answer_v2("이 문장 글자수랑 단어수 알려줘"))
print(answer_v2("2*(3+4) 계산해줘"))
print(answer_v2("지금 몇 시야?"))

tool_name : text_stats, args : {'text': '이 문장 글자수랑 단어수 알려줘'}
호출한 tool은? : Tool(name='text_stats', description='문자열의 글자 수/단어 수를 계산합니다.', fn=<function <lambda> at 0x7d405dd7f4c0>)
(Tool 사용: text_stats) 결과: {'chars': 17, 'words': 5}
tool_name : calc, args : {'expr': '2*(3+4)'}
호출한 tool은? : Tool(name='calc', description="수식을 계산합니다. 예: '2*(3+4)'", fn=<function <lambda> at 0x7d405e7a4ea0>)
(Tool 사용: calc) 결과: 14
tool_name : now, args : {}
호출한 tool은? : Tool(name='now', description='현재 날짜/시간을 문자열로 반환합니다.', fn=<function <lambda> at 0x7d405e7a4180>)
(Tool 사용: now) 결과: 2026-04-13 11:53:29


In [ ]:
# 이해를 위한, MCP 실행 코드 1)
from openai import OpenAI
from dataclasses import dataclass
from typing import Callable, Dict, Any
from datetime import datetime
import json
import math
import re
import os

# OpenAI의 API를 사용하게 해주는 client 요청자다. 근데 그게 객체이다 클래스이다.
client = OpenAI(api_key="")

@dataclass
class Tool:
    name: str
    description: str
    fn: Callable[..., Any]

TOOLS: Dict[str, Tool] = {}

def register_tool(tool: Tool):
    TOOLS[tool.name] = tool

# 실제 실행 함수들 / 위와 구조는 똑같고, 오픈ai 사용하기 위해 해놓음
def calc(expr: str):
    safe_globals = {"__builtins__": {}}
    safe_locals = {"math": math}
    result = eval(expr, safe_globals, safe_locals)
    return {"expr": expr, "result": result}

def now():
    return {
        "datetime": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "date": datetime.now().strftime("%Y-%m-%d"),
        "time": datetime.now().strftime("%H:%M:%S")
    }

def text_stats(text: str):
    return {
        "chars": len(text),
        "words": len([w for w in text.split() if w])
    }

register_tool(Tool(
    name="calc",
    description="수학 식을 계산합니다. 예: 2*(3+4), 10/2, 3.5*8",
    fn=calc
))

register_tool(Tool(
    name="now",
    description="현재 날짜와 시간을 반환합니다.",
    fn=now
))

register_tool(Tool(
    name="text_stats",
    description="문자열의 글자 수와 단어 수를 계산합니다.",
    fn=text_stats
))
# ---- openai_tools 라는 것만 다름 ----
# OpenAI API에 넘길 tool schema
openai_tools = [
    {
        "type": "function",
        "name": "calc",
        "description": "수학 식을 계산합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "expr": {
                    "type": "string",
                    "description": "계산할 수식. 예: 2*(3+4)"
                }
            },
            "required": ["expr"],
            "additionalProperties": False
        }
    },
    {
        "type": "function",
        "name": "now",
        "description": "현재 날짜와 시간을 반환합니다.",
        "parameters": {
            "type": "object",
            "properties": {},
            "required": [],
            "additionalProperties": False
        }
    },
    {
        "type": "function",
        "name": "text_stats",
        "description": "문자열의 글자 수와 단어 수를 계산합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "text": {
                    "type": "string",
                    "description": "분석할 문자열"
                }
            },
            "required": ["text"],
            "additionalProperties": False
        }
    }
]

def tool_call(tool_name: str, args: Dict[str, Any]) -> Any:
    if tool_name not in TOOLS:
        return {"error": f"unknown tool: {tool_name}"}
    try:
        tool = TOOLS[tool_name]
        return tool.fn(**args) if args else tool.fn()
    except Exception as e:
        return {"error": str(e)}

# MCP를 사용하기 위해 코드를 만들어 놓음

In [ ]:
# 이해를 위한, MCP 실행 코드 1)
def answer_with_gpt(user_text: str, model: str = "gpt-4.1") -> str:
    # 1차 호출: GPT가 tool 선택. ai 가 사용할 툴(calc, now, text) 중 선택
    response = client.responses.create(
        model=model,
        input=user_text,
        tools=openai_tools, # 위에 만든 openai_tools. tools가 있는데, tools에는 calc, now, text_stat
        tool_choice="auto"  # tool 은 적절히 판단해서 골라와라
    )

    # tool call 찾기
    tool_calls = [item for item in response.output if item.type == "function_call"]

    # tool을 안 썼으면 일반 답변 반환
    if not tool_calls:
        return response.output_text
    print(f"tool_")

    # 여기서는 수업용으로 첫 번째 tool call만 처리
    call = tool_calls[0]
    tool_name = call.name
    args = json.loads(call.arguments) if call.arguments else {}

    print("선택된 tool:", tool_name)
    print("전달 인자:", args)

    # 실제 함수 실행
    result = tool_call(tool_name, args)
    print("실행 결과:", result)

    # 2차 호출: tool 결과를 다시 모델에 전달. 툴을 기준으로 재생성
    followup = client.responses.create(
        model=model,
        previous_response_id=response.id,
        input=[
            {
                "type": "function_call_output",
                "call_id": call.call_id,
                "output": json.dumps(result, ensure_ascii=False)
            }
        ],
        tools=openai_tools,
        tool_choice="auto"
    )

    return followup.output_text


# 빠른 테스트
tests = [
    "2*(3+4) 계산해줘",
    "지금 몇 시야?",
    "이 문장 글자수랑 단어수 알려줘: 안녕하세요 저는 김진산입니다",
    "파이썬이 뭐야?"
]

for t in tests:
    print("Q:", t)
    print("A:", answer_with_gpt(t))
    print("---")

Q: 2*(3+4) 계산해줘
tool_
선택된 tool: calc
전달 인자: {'expr': '2*(3+4)'}
실행 결과: {'expr': '2*(3+4)', 'result': 14}
A: 2 × (3 + 4)의 계산 결과는 14입니다.
---
Q: 지금 몇 시야?
tool_
선택된 tool: now
전달 인자: {}
실행 결과: {'datetime': '2026-04-13 12:10:49', 'date': '2026-04-13', 'time': '12:10:49'}
A: 지금은 2026년 4월 13일 12시 10분 49초입니다.
---
Q: 이 문장 글자수랑 단어수 알려줘: 안녕하세요 저는 김진산입니다
tool_
선택된 tool: text_stats
전달 인자: {'text': '안녕하세요 저는 김진산입니다'}
실행 결과: {'chars': 15, 'words': 3}
A: 이 문장은 글자수 15자, 단어수 3개입니다.
---
Q: 파이썬이 뭐야?
A: 파이썬(Python)은 쉽고 간결한 문법을 가지면서도 강력한 기능을 제공하는 프로그래밍 언어입니다. 다양한 분야(웹 개발, 데이터 분석, 인공지능, 머신러닝, 게임 개발 등)에서 널리 사용되고 있습니다.

파이썬의 특징은 다음과 같습니다:

1. **간결한 문법**: 읽고 쓰기 쉬운 코드 덕분에 초보자도 쉽게 배울 수 있습니다.
2. **다양한 라이브러리**: 이미 만들어진 많은 기능(예: 수학, 통계, 그래픽, 웹 등)을 손쉽게 사용할 수 있습니다.
3. **플랫폼 독립적**: 윈도우, 맥, 리눅스 등 다양한 운영체제에서 실행됩니다.
4. **커뮤니티 지원**: 전 세계적으로 많은 개발자들이 사용하고, 다양한 자료와 예제가 많습니다.

간단한 예시 코드:
```python
print("Hello, world!")
```
위 코드는 "Hello, world!"를 화면에 출력합니다.

궁금한 점이나 더 알고 싶은 부분이 있으면 말씀해주세요!
---
